# Week 6 Homework

***Due (pushed to your GitHub branch) on 10/24 by 11:59 pm***

## STAR two-pass alignment

Build the STAR SA:

In [3]:
%mkdir -p homework/alignment/star/
%mkdir ../2_genome_exploration/genome/homework/star

!STAR \
--runThreadN 32 \
--runMode genomeGenerate \
--genomeDir ../2_genome_exploration/genome/homework/star \
--genomeFastaFiles ../2_genome_exploration/genome/my_genome.fna \
--sjdbGTFfile ../2_genome_exploration/genome/my_annotations.gtf \
--sjdbOverhang 89 \
--genomeSAindexNbases 13

Use STAR and ***all*** of your FASTQs for a first-pass alignment. Be sure to update the `sjdbOverhang` parameter to a value that is appropriate for your dataset. Rename the log file so that you can compare the logs from the first and second alignments:

In [1]:
!STAR \
--runThreadN 32 \
--runMode alignReads \
--genomeDir ../2_genome_exploration/genome/homework/star \
--readFilesManifest homework_manifest.tsv \
--readFilesCommand zcat \
--outSAMtype BAM Unsorted \
--outSAMunmapped Within \
--outFileNamePrefix homework/alignment/star/

	STAR --runThreadN 32 --runMode alignReads --genomeDir ../2_genome_exploration/genome/homework/star --readFilesManifest homework_manifest.tsv --readFilesCommand zcat --outSAMtype BAM Unsorted --outSAMunmapped Within --outFileNamePrefix homework/alignment/star/
	STAR version: 2.7.10b   compiled: 2022-11-01T09:53:26-04:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Nov 01 22:28:34 ..... started STAR run
Nov 01 22:28:41 ..... loading genome
Nov 01 22:29:52 ..... started mapping
Nov 01 23:31:51 ..... finished mapping
Nov 01 23:31:56 ..... finished successfully


In [2]:
!mv homework/alignment/star/Log.final.out homework/alignment/star/first-pass.final.out

Use STAR, all of your FASTQs, and `SJ.out.tab` for the second-pass alignment. You will likely need to make your own `manifest.tsv`:

In [3]:
!STAR \
--runThreadN 32 \
--runMode alignReads \
--genomeDir ../2_genome_exploration/genome/homework/star \
--readFilesManifest homework_manifest.tsv \
--readFilesCommand zcat \
--outSAMtype BAM Unsorted \
--outSAMunmapped Within \
--outSAMattributes NH HI AS nM RG \
--outFileNamePrefix homework/alignment/star/ \
--sjdbFileChrStartEnd homework/alignment/star/SJ.out.tab

	STAR --runThreadN 32 --runMode alignReads --genomeDir ../2_genome_exploration/genome/homework/star --readFilesManifest homework_manifest.tsv --readFilesCommand zcat --outSAMtype BAM Unsorted --outSAMunmapped Within --outSAMattributes NH HI AS nM RG --outFileNamePrefix homework/alignment/star/ --sjdbFileChrStartEnd homework/alignment/star/SJ.out.tab
	STAR version: 2.7.10b   compiled: 2022-11-01T09:53:26-04:00 :/home/dobin/data/STAR/STARcode/STAR.master/source
Nov 01 23:42:55 ..... started STAR run
Nov 01 23:43:04 ..... loading genome
Nov 01 23:43:35 ..... inserting junctions into the genome indices
Nov 01 23:45:50 ..... started mapping
Nov 02 00:48:30 ..... finished mapping
Nov 02 00:48:35 ..... finished successfully


In [4]:
!mv homework/alignment/star/Log.final.out homework/alignment/star/second-pass.final.out

Update this block to briefly describe the differences between the logs for the first and second alignments.   
The number of splices was higher for the second alignment by around 5 million and the number of annotated splices is much closer to the total number. The second alignment has a slightly lower percentage of uniquely matched reads (-0.1%) and a higher percentage of reads mapped to multiple loci (+0.2%). There were the same percentage of reads left unmapped for other unlisted reasons and fewer reads unmapped due to being too short (-0.09%).

## HISAT alignment

(***Note***: you may need Dr. Wheeler to build the HISAT index of your genome. Ask him for assistance when you get to this part of the assignment.)

Use a Bash `while` loop to align all of your FASTQs and save the output files using the same names as the input FASTQ:

In [ ]:
skip

## Snakemake Pipeline

In contrast to last week (FASTQ files), there aren't as many decisions that need to be made during alignment, so using a notebook to control and report code isn't as necessary. As you saw in in [6_alignment](6_alignment.ipynb), file and directory management can get complicated during alignment. This is a case where Snakemake scripting shines.

1. Copy your `Snakefile` and `config.yaml` from `5_fastq` to `6_alignment`. 
2. In the `Snakefile`, add the following rules:  
    a. `fetch_genome` - using a variation from the code from `2_genome_exploration`, download your reference genome and annotations.  
    b. `star_sa` - use `STAR` to produce the suffix array of the reference genome.  
    c. `star_align_first` - use `STAR` to perform first-pass alignment with all FASTQ files.  
    d. `star_align_second` - use `STAR` to perform second-pass alignment with all FASTQ files.  
    e. `hisat_align` - use `HISAT` to align all FASTQ files.  
    f. `report` - use `multiqc` to aggregate all QC data into a single report.

This `Snakefile` should not use any `wrappers` because we already have the relevant software installed in the `biol343` conda environment. The pipeline should run to completion when any instructor or classmate runs `snakemake --use-conda`.